# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Limit Order Book Reconstruction & Market Microstructure

---
*Nota Institucional: Este notebook modela la dinámica de reconstrucción asíncrona de un Limit Order Book (LOB) a partir de un flujo de eventos nivel 2 (Tick-by-Tick). En un entorno de producción corporativo de ultra-baja-latencia, el motor de matching y agregación del estado puro del libro (L2) se programaría invariablemente en C# / C++ interactuando directamente con sockets multicast. Python, apoyado por librerías asíncronas (`asyncio`) y JIT (`Numba` o `Cython`), asume la orquestación a posteriori, el análisis estadístico y la persistencia en arquitecturas columnares.*

In [ ]:
import pandas as pd
import numpy as np
import asyncio
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("crest")

### 1. Ingestión y Procesamiento Asíncrono del Flujo Tick-by-Tick

El siguiente corrutinaje simula la absorción de un *stream* volumétrico reconstruyendo la profundidad de mercado iterativamente.

In [ ]:
class AsyncLOBProcessor:
    def __init__(self):
        self.bids = {} # Price -> Size
        self.asks = {} # Price -> Size
        self.spread_history = []
        self.mid_price_history = []
        self.timestamps = []
        
    async def process_event(self, event):
        """Asynchronous handler for L2 market data events"""
        side = event['side']
        price = event['price']
        size = event['size']
        e_type = event['event_type']
        ts = event['timestamp']
        
        book = self.bids if side == 'Bid' else self.asks
        
        if e_type == 1: # Submission
            book[price] = book.get(price, 0) + size
        elif e_type in [2, 3]: # Cancellation or Execution
            if price in book:
                book[price] -= size
                if book[price] <= 0:
                    del book[price]
                    
        self._record_metrics(ts)
        await asyncio.sleep(0) # Yield control to event loop
        
    def _record_metrics(self, ts):
        if not self.bids or not self.asks: return
        
        best_bid = max(self.bids.keys())
        best_ask = min(self.asks.keys())
        
        if best_bid < best_ask:
            spread = best_ask - best_bid
            mid = (best_ask + best_bid) / 2
            self.spread_history.append(spread)
            self.mid_price_history.append(mid)
            self.timestamps.append(ts)

async def run_simulation(df):
    processor = AsyncLOBProcessor()
    # Creating async tasks for batch processing simulation
    tasks = [processor.process_event(row) for _, row in df.iterrows()]
    await asyncio.gather(*tasks)
    return processor

In [ ]:
# We load the deterministic dataset (RANDOM_SEED=42)
# Note: In production this is streamed directly from memory/socket, not CSV.
df = pd.read_csv('../data/lob_events_synthetic.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp')

# Ejecutar ciclo asincrónico (event loop handling for Jupyter)
processor = await run_simulation(df)

### 2. Extracción de Microestructura y Análisis Descriptivo
Con el estado del libro reconstruido en memoria, podemos extraer métricas críticas como la evolución del spread efectivo y la tendencia estocástica del precio *mid*.

In [ ]:
metrics_df = pd.DataFrame({
    'timestamp': processor.timestamps,
    'mid_price': processor.mid_price_history,
    'spread': processor.spread_history
})
metrics_df.set_index('timestamp', inplace=True)
metrics_df = metrics_df.resample('1S').mean().ffill()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(metrics_df.index, metrics_df['mid_price'], color='cyan', alpha=0.8)
ax1.set_title('Evolución del Mid-Price (Nivel 2 LOB Reconstruido)', fontsize=14, pad=15)
ax1.set_ylabel('Precio ($)', fontsize=12)
ax1.grid(True, alpha=0.2)

ax2.plot(metrics_df.index, metrics_df['spread'], color='coral', alpha=0.9)
ax2.axhline(y=metrics_df['spread'].mean(), color='white', linestyle='--', alpha=0.5, label='Media Histórica')
ax2.set_title('Dinámica del Bid-Ask Spread Efectivo', fontsize=14, pad=15)
ax2.set_ylabel('Spread ($)', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()